# 04 — Perbandingan Baseline dan IndoBERT

**Tujuan:** membandingkan TF-IDF + Logistic Regression dengan IndoBERT pada validation yang sama, lalu memeriksa hasil test bila sudah final.

**Prasyarat:** `out/baseline_val.json`, `models/indobert_real/val_metrics.json`, dan `models/indobert_aug/val_metrics.json`.

**Estimasi waktu:** 1 menit.

## Glosarium

- **Baseline**: model sederhana sebagai acuan.
- **Fine-tuning**: melatih model bahasa pada tugas khusus.
- **Selisih F1**: perbedaan F1 kelas bully, bukan accuracy.
- **Test set**: hanya untuk laporan akhir, bukan pemilihan model.

## 1. Muat hasil

Notebook hanya menampilkan artefak yang ada. Kalau salah satu belum ada, ia
memberi pesan, bukan mengarang angka. Threshold tiap model berasal dari
validation masing-masing.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from cyberbully.config import MODELS_DIR, OUT_DIR

baseline = json.loads((OUT_DIR / "baseline_val.json").read_text()) if (OUT_DIR / "baseline_val.json").exists() else None
MODEL_DIRS = {
    "A": MODELS_DIR / "indobert_real",
    "B": MODELS_DIR / "indobert_aug",
}
indobert = {}
for key, directory in MODEL_DIRS.items():
    path = directory / "val_metrics.json"
    indobert[key] = json.loads(path.read_text()) if path.exists() else None
print("baseline:", baseline is not None)
print({key: value is not None for key, value in indobert.items()})


## 2. Tabel validation

Tabel memuat precision, recall, F1, dan accuracy kelas bully. Perhatikan juga
threshold: threshold rendah menaikkan recall tetapi bisa menambah false positive.
Perbandingan hanya bermakna jika split dan seed sama.

In [ ]:
rows = []
if baseline is not None:
    rows.append({"model": "TF-IDF + LogReg", "threshold": baseline["threshold"], **{k: baseline["val"][k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
for key, label in (("A", "IndoBERT A — real"), ("B", "IndoBERT B — asli+sintetis")):
    payload = indobert[key]
    if payload is not None:
        rows.append({"model": label, "threshold": payload["threshold"], **{k: payload[k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
print(pd.DataFrame(rows).set_index("model").round(3).to_string() if rows else "Belum ada hasil validation.")


## 3. Grafik metrik

Grafik membantu melihat apakah keunggulan konsisten di precision, recall, dan F1.
Selisih di bawah 0,01–0,02 F1 sebaiknya tidak langsung disebut peningkatan berarti;
lihat juga error analysis dan ukuran data.

In [ ]:
if rows:
    metrics = ["precision_bully", "recall_bully", "f1_bully"]
    labels = [row["model"] for row in rows]
    width = 0.35
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for offset, row in enumerate(rows):
        positions = [i + offset * width for i in range(len(metrics))]
        ax.bar(positions, [row[key] for key in metrics], width=width, label=row["model"])
    ax.set_xticks([i + width / 2 for i in range(len(metrics))], ["precision", "recall", "F1"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("skor kelas bully")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
else:
    print("Dilewati: belum ada hasil.")

## 4. Hasil test (bila sudah final)

Test hanya dievaluasi sekali per model setelah model dan threshold dibekukan.
Kalau file hasil test belum ada, bagian ini menyatakan belum final. Jangan memakai
test untuk memilih model, threshold, atau hyperparameter.

In [ ]:
test_rows = []
if (OUT_DIR / "baseline_test.json").exists():
    payload = json.loads((OUT_DIR / "baseline_test.json").read_text())
    test_rows.append({"model": "TF-IDF + LogReg", "threshold": payload["threshold"], **{k: payload["metrics"][k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
for key, label in (("A", "IndoBERT A — real"), ("B", "IndoBERT B — asli+sintetis")):
    path = OUT_DIR / f"model_{key.lower()}" / "test_metrics.json"
    if path.exists():
        payload = json.loads(path.read_text())
        test_rows.append({"model": label, "threshold": payload["threshold"], **{k: payload[k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
print(pd.DataFrame(test_rows).set_index("model").round(3).to_string() if test_rows else "Hasil test belum ada; laporan belum final.")


## Ringkasan

- Perbandingan memakai validation yang sama dan metrik yang sama.
- F1 bully adalah metrik utama; precision/recall menjelaskan arah trade-off.
- Test hanya dibaca sekali di akhir dan tidak dipakai memilih model.